# esp-bridge 実機テスト
Colab で ActiveHaptic をビルドし、手元の ESP32 に**ブラウザ経由で**書き込んで、シリアル出力を Colab で読む。

1. 上から順に実行する(ESP-IDF のインストールとビルドで5分ほど)。
2. 「▶ ESP32 ブリッジを開く」を **Chrome か Edge** で開き、ESP32 を USB でつないで「ESP32 に接続」を押す。
3. 最後のセルで書き込む。タブに確認が出たら「書き込む」を押す。

In [ ]:
!rm -rf /content/esp-bridge && git clone -q https://github.com/hasevr/esp-bridge /content/esp-bridge

In [ ]:
import sys; sys.path.insert(0, '/content/esp-bridge')
import esp_bridge; esp_bridge.start()

## ESP-IDF のインストールと ActiveHaptic のビルド

In [ ]:
%%bash
# 1. ESP-IDF の取得とインストール
set -e
T0=$(date +%s)
apt-get -qq update > /dev/null && apt-get -qq install -y flex bison gperf ninja-build ccache libffi-dev libssl-dev dfu-util libusb-1.0-0 python3-venv python$(python3 -c 'import sys;print(f"{sys.version_info[0]}.{sys.version_info[1]}")')-venv > /dev/null
T1=$(date +%s); echo "### apt: $((T1-T0)) s"
cd /content
[ -d esp-idf ] || git clone -q -b v5.5.1 --depth 1 --recursive --shallow-submodules https://github.com/espressif/esp-idf.git
T2=$(date +%s); echo "### clone: $((T2-T1)) s"
export IDF_TOOLS_PATH=/content/.espressif
rm -rf /content/.espressif/python_env
./esp-idf/install.sh esp32 > install.log 2>&1 || { tail -40 install.log; exit 1; }
T3=$(date +%s); echo "### install: $((T3-T2)) s"
du -sh /content/esp-idf /content/.espressif

In [ ]:
%%bash
# 2. ActiveHaptic プロジェクトの用意 (Scrapbox 手順 5-1 と同じ CMakeLists.txt)
set -e
cd /content
rm -rf ActiveHaptic
git clone -q https://github.com/hasevr/ICTEx5ActiveHaptic ActiveHaptic
cat > ActiveHaptic/main/CMakeLists.txt <<'EOF'
idf_component_register(SRCS "main.c"
                    INCLUDE_DIRS "."
                    REQUIRES esp_driver_uart driver spi_flash esp_adc esp_timer)
EOF
grep CONFIG_FREERTOS_HZ= ActiveHaptic/sdkconfig

In [ ]:
%%bash
# 3. ビルド
set -e
export IDF_TOOLS_PATH=/content/.espressif
. /content/esp-idf/export.sh > /dev/null
cd /content/ActiveHaptic
T0=$(date +%s)
# set-target は sdkconfig を作り直して Tick 1000Hz が消えるので使わない
idf.py build > ../build.log 2>&1 || { tail -60 ../build.log; exit 1; }
echo "### build: $(( $(date +%s)-T0 )) s"
tail -5 ../build.log

## 書き込みと確認
タブで「書き込む」を押すまで待つ。

In [ ]:
!esp status
!esp flash /content/ActiveHaptic

In [ ]:
!esp log --since-flash -n 40

In [ ]:
# 力センサを押しながら実行すると ADC の値が変わるはず
!esp reset
import time; time.sleep(3)
!esp log -n 20